# Projeto ArPrev — 02 · Features dos Dados Recentes
**ArPrev — Foco na Antecipação de Condições Inadequadas**  
*Mogi das Cruzes / SP — Campus UBC | Trilha B (Qualidade do Ar)*

Segundo passo do fluxo operacional de previsão **+1h a partir de agora**. Este notebook lê a cópia criada em [`01_Coleta_Dados_Recentes.ipynb`](01_Coleta_Dados_Recentes.ipynb) e calcula:

1. O **IQAr** de cada hora (metodologia CETESB 2025: médias móveis por poluente e maior subíndice).
2. As features de curto prazo de ozônio e PM2,5: **média das últimas 3 horas** e **variação em 2 horas**.
3. A **linha mais recente completa**, que é a entrada do modelo.

Execute as células em ordem. Se a cópia não existir, rode antes o notebook 01.

## 1. Carregar a cópia dos dados recentes

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
import yaml

ROOT = Path.cwd()
if not (ROOT / 'config' / 'params.yaml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'params.yaml').is_file():
    raise FileNotFoundError('Não encontrei config/params.yaml. Abra o notebook a partir da raiz do projeto.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.aplicacao.Prever_Proxima_Hora import (
    COLUNAS_CONTEXTO_CURTO_PRAZO,
    _carregar_artefato_modelo,
    _preparar_linha_mais_recente,
    MODELO_FINAL_PATH,
)
from src.transformacao.Calcular_Alvo_IQAr import calcular_iqar
from src.transformacao.Features_Temporais import adicionar_resumos_curto_prazo

with (ROOT / 'config' / 'params.yaml').open('r', encoding='utf-8') as arquivo:
    config = yaml.safe_load(arquivo)

caminho_copia = ROOT / config['application']['recent_snapshot']
if not caminho_copia.is_file():
    raise FileNotFoundError(f'Execute antes o notebook 01. Não encontrei: {caminho_copia}')

dados = pd.read_csv(caminho_copia, parse_dates=['time'])
print(f'{len(dados)} horas, de {dados["time"].iloc[0]} até {dados["time"].iloc[-1]}')

96 horas, de 2026-10-17 00:00:00 até 2026-10-20 23:00:00


## 2. Calcular o IQAr e as features temporais

As primeiras horas ficam sem valor porque o IQAr usa janelas de até 24 horas (PM10, PM2,5 e SO₂). Por isso o histórico precisa ter vários dias.

In [2]:
com_iqar = calcular_iqar(dados)
com_features = adicionar_resumos_curto_prazo(com_iqar, COLUNAS_CONTEXTO_CURTO_PRAZO)

colunas_resumo = [
    'time', 'iqar', 'iqar_ozone', 'iqar_pm2_5',
    'media_3h_ozone', 'media_3h_pm2_5', 'delta_2h_ozone', 'delta_2h_pm2_5',
]
com_features[colunas_resumo].tail(8)

,time,iqar,iqar_ozone,iqar_pm2_5,media_3h_ozone,media_3h_pm2_5,delta_2h_ozone,delta_2h_pm2_5
88,2026-10-20 16:00:00,28.80,28.80,11.100000,74.666667,7.466667,-6.0,0.7
89,2026-10-20 17:00:00,28.75,28.75,10.953333,70.000000,7.766667,-13.0,0.5
90,2026-10-20 18:00:00,28.15,28.15,10.800000,63.000000,7.733333,-16.0,-0.4
91,2026-10-20 19:00:00,26.95,26.95,10.713333,55.000000,7.466667,-16.0,-1.0
92,2026-10-20 20:00:00,25.25,25.25,10.666667,47.333333,7.033333,-15.0,-0.7
93,2026-10-20 21:00:00,23.20,23.20,10.586667,40.666667,6.733333,-12.0,-0.5
94,2026-10-20 22:00:00,21.10,21.10,10.533333,36.666667,6.633333,-5.0,0.0
95,2026-10-20 23:00:00,19.15,19.15,10.493333,35.666667,6.600000,2.0,0.1


## 3. Selecionar a linha de entrada do modelo

O modelo recebe exatamente as features congeladas no artefato. A linha escolhida é a **última hora com todas as features preenchidas**; é ela que descreve "agora".

In [3]:
artefato = _carregar_artefato_modelo(ROOT / MODELO_FINAL_PATH)
features = list(artefato['features'])
entrada = _preparar_linha_mais_recente(dados, features)

print(f'Features exigidas pelo modelo: {len(features)}')
print(f'Hora usada como "agora" (t): {entrada["time"].iloc[0]}')
entrada.T

Features exigidas pelo modelo: 15
Hora usada como "agora" (t): 2026-10-20 23:00:00


,95
iqar_pm10,7.23
iqar_pm2_5,10.493333
iqar_carbon_monoxide,0.414405
iqar_nitrogen_dioxide,0.88
iqar_sulphur_dioxide,2.125
iqar_ozone,19.15
temperature_2m,13.3
relative_humidity_2m,96
precipitation,0.0
wind_speed_10m,7.9


## Próximo passo

Abra [`03_Previsao_Proxima_Hora.ipynb`](03_Previsao_Proxima_Hora.ipynb) para obter a probabilidade e a decisão para a hora seguinte.